In [2]:
import time
import torch
import torch.nn as nn

SEED = 5996
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

INPUT_DIM = 256
HIDDEN = 512
NUM_CLASSES = 10
BATCH_SIZE = 128
STEPS = 50

X = torch.randn(8000, INPUT_DIM)
y = torch.randint(0, NUM_CLASSES, (8000,))

loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X, y),
    batch_size=BATCH_SIZE,
    shuffle=True
)

class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(INPUT_DIM, HIDDEN)
        self.fc2 = nn.Linear(HIDDEN, HIDDEN)
        self.fc3 = nn.Linear(HIDDEN, NUM_CLASSES)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

Device: cuda


In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No CUDA GPU detected")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


1.Tensor Creation: CPU vs GPU

In [3]:
shape = (3000, 3000)

torch.cuda.synchronize()
start = time.perf_counter()

x = torch.randn(shape)
x = x.to(device)

torch.cuda.synchronize()
cpu_transfer_time = time.perf_counter() - start


torch.cuda.synchronize()
start = time.perf_counter()

x = torch.randn(shape, device=device)

torch.cuda.synchronize()
gpu_creation_time = time.perf_counter() - start

print("CPU creation + transfer:", cpu_transfer_time)
print("Direct GPU creation:", gpu_creation_time)

CPU creation + transfer: 0.09506555600000866
Direct GPU creation: 0.012764800000013565


In [4]:
def train_model(model, use_amp=False):

    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    start = time.perf_counter()
    data_iter = iter(loader)

    for step in range(STEPS):

        try:
            xb, yb = next(data_iter)
        except StopIteration:
            data_iter = iter(loader)
            xb, yb = next(data_iter)

        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()

        if use_amp and torch.cuda.is_available():
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                output = model(xb)
                loss = criterion(output, yb)
        else:
            output = model(xb)
            loss = criterion(output, yb)

        loss.backward()
        optimizer.step()

    if torch.cuda.is_available():
        torch.cuda.synchronize()
        memory = torch.cuda.max_memory_allocated() / 1024**2
    else:
        memory = None

    elapsed = time.perf_counter() - start

    return elapsed, memory, loss.item()

2.Weight Intialization

In [5]:
torch.manual_seed(SEED)
default_model = SimpleMLP()

default_result = train_model(default_model)


torch.manual_seed(SEED)
xavier_model = SimpleMLP()

for layer in xavier_model.modules():
    if isinstance(layer, nn.Linear):
        nn.init.xavier_uniform_(layer.weight)
        nn.init.zeros_(layer.bias)

xavier_result = train_model(xavier_model)

print("Default:", default_result)
print("Xavier:", xavier_result)

Default: (0.6405302189999702, 59.33349609375, 2.294248580932617)
Xavier: (0.12074489499997298, 62.38134765625, 2.324660301208496)


3.Activation Checkpointing

In [6]:
torch.manual_seed(SEED)
default_model = SimpleMLP()

default_result = train_model(default_model)


torch.manual_seed(SEED)
xavier_model = SimpleMLP()

for layer in xavier_model.modules():
    if isinstance(layer, nn.Linear):
        nn.init.xavier_uniform_(layer.weight)
        nn.init.zeros_(layer.bias)

xavier_result = train_model(xavier_model)

print("Default:", default_result)
print("Xavier:", xavier_result)

Default: (0.12814220500001738, 62.38134765625, 2.294248580932617)
Xavier: (0.11857490899996037, 62.38134765625, 2.324660301208496)


In [8]:
from torch.utils.checkpoint import checkpoint

class CheckpointMLP(SimpleMLP):
    def forward(self, x):

        def block(x):
            x = torch.relu(self.fc1(x))
            x = torch.relu(self.fc2(x))
            return x

        x.requires_grad_(True)
        x = checkpoint(
            block,
            x,
            use_reentrant=False
        )

        return self.fc3(x)

In [9]:
torch.manual_seed(SEED)
normal_result = train_model(SimpleMLP())

torch.manual_seed(SEED)
checkpoint_result = train_model(CheckpointMLP())

print("Normal:", normal_result)
print("Checkpointing:", checkpoint_result)

Normal: (0.17485188199998447, 65.42919921875, 2.294248580932617)
Checkpointing: (0.2881881139999791, 66.55419921875, 2.294248580932617)


4.Gradient Accumulation

In [10]:
def train_with_accumulation(model, accumulation_steps=2):

    model = model.to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-3
    )

    criterion = nn.CrossEntropyLoss()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    start = time.perf_counter()

    data_iter = iter(loader)
    optimizer.zero_grad()

    for step in range(STEPS):

        try:
            xb, yb = next(data_iter)
        except StopIteration:
            data_iter = iter(loader)
            xb, yb = next(data_iter)

        xb, yb = xb.to(device), yb.to(device)

        output = model(xb)

        loss = criterion(output, yb)
        loss = loss / accumulation_steps

        loss.backward()

        if (step + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

    if torch.cuda.is_available():
        torch.cuda.synchronize()
        memory = torch.cuda.max_memory_allocated() / 1024**2
    else:
        memory = None

    elapsed = time.perf_counter() - start

    return elapsed, memory, loss.item() * accumulation_steps

In [11]:
torch.manual_seed(SEED)
accum_result = train_with_accumulation(
    SimpleMLP(),
    accumulation_steps=2
)

print("Gradient accumulation:", accum_result)

Gradient accumulation: (0.16717769699999963, 66.65771484375, 2.2961926460266113)


5.Mixed Preciison

In [12]:
torch.manual_seed(SEED)

fp32_result = train_model(
    SimpleMLP(),
    use_amp=False
)

torch.manual_seed(SEED)

amp_result = train_model(
    SimpleMLP(),
    use_amp=True
)

print("FP32:", fp32_result)
print("Mixed Precision:", amp_result)

FP32: (0.12211467400004494, 66.42919921875, 2.294248580932617)
Mixed Precision: (0.3232091129999617, 66.4267578125, 2.29229736328125)


In [13]:
import pandas as pd

results = pd.DataFrame([
    ["Weight Init - Default", *default_result],
    ["Weight Init - Xavier", *xavier_result],
    ["No Checkpoint", *normal_result],
    ["Checkpointing", *checkpoint_result],
    ["Gradient Accumulation", *accum_result],
    ["FP32", *fp32_result],
    ["Mixed Precision", *amp_result]
], columns=[
    "Experiment",
    "Time (sec)",
    "GPU Memory (MB)",
    "Final Loss"
])

results

,Experiment,Time (sec),GPU Memory (MB),Final Loss
0,Weight Init - Default,0.128142,62.381348,2.294249
1,Weight Init - Xavier,0.118575,62.381348,2.324660
2,No Checkpoint,0.174852,65.429199,2.294249
3,Checkpointing,0.288188,66.554199,2.294249
4,Gradient Accumulation,0.167178,66.657715,2.296193
5,FP32,0.122115,66.429199,2.294249
6,Mixed Precision,0.323209,66.426758,2.292297


In [14]:
print("CPU + GPU transfer:", cpu_transfer_time)
print("Direct GPU creation:", gpu_creation_time)

CPU + GPU transfer: 0.09506555600000866
Direct GPU creation: 0.012764800000013565


Five optimization techniques were evaluated under a small controlled PyTorch setup. Direct GPU tensor creation was compared with CPU creation followed by transfer to measure data-transfer overhead. Xavier initialization was compared with PyTorch's default initialization using final training loss. Activation checkpointing and gradient accumulation were evaluated for their effects on execution time and GPU memory, while mixed precision was compared with FP32 training. The results illustrate that optimization techniques involve trade-offs between training speed, memory usage, and convergence behavior.